# March Machine Learning Mania 2025 - Kaggle Competition Template

In [27]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# 📂 Define the data path
data_path = "data/"

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np

# 📂 Define the data path
data_path = "data/"  # Replace with your actual data folder path

# Load CSV files into DataFrames
conference_tourney_games = pd.read_csv(data_path + 'MConferenceTourneyGames.csv')
teams = pd.read_csv(data_path + 'MTeams.csv')
team_conferences = pd.read_csv(data_path + 'MTeamConferences.csv')
regular_season_detailed_results = pd.read_csv(data_path + 'MRegularSeasonDetailedResults.csv')

# Step 1: Start with a season-based dataset as the base
mens_data = team_conferences  # Includes 'Season' and 'TeamID'

# Step 2: Merge team details from the `teams` dataset
mens_data = pd.merge(mens_data, teams, on="TeamID", how="left")

# Step 3: Handle regular season detailed results
# Merge with Winning Teams
regular_season_winners = pd.merge(
    regular_season_detailed_results, teams, left_on="WTeamID", right_on="TeamID", how="left"
).rename(columns={"TeamName": "WTeamName"})

# Merge with Losing Teams
regular_season_losers = pd.merge(
    regular_season_winners, teams, left_on="LTeamID", right_on="TeamID", how="left"
).rename(columns={"TeamName": "LTeamName"})

# Drop duplicate 'TeamID' columns from merges
regular_season_losers.drop(columns=["TeamID_x", "TeamID_y"], inplace=True)

# Add the combined regular season data to `mens_data`
mens_data = pd.merge(
    mens_data, 
    regular_season_losers, 
    left_on=["Season", "TeamID"], 
    right_on=["Season", "WTeamID"], 
    how="left"
)

# Step 4: Merge additional datasets as needed
mens_data = pd.merge(mens_data, conference_tourney_games, on=["Season", "WTeamID"], how="left")

# Step 5: Handle missing values
mens_data.fillna({
    "ConfAbbrev": "Unknown",  # Unknown conference
    "DayNum": -1,            # Invalid day number for missing days
    "WScore": 0,             # Replace missing winning scores with 0
    "LScore": 0,             # Replace missing losing scores with 0
    "WLoc": "N",             # Assume neutral location for missing data
    "NumOT": 0               # Replace missing overtime counts with 0
}, inplace=True)

# Fill missing team names with "Unknown"
mens_data['WTeamName'].fillna("Unknown", inplace=True)
mens_data['LTeamName'].fillna("Unknown", inplace=True)

# Step 6: Drop or consolidate redundant columns
mens_data.drop(columns=["ConfAbbrev_y", "FirstD1Season_y", "LastD1Season_y"], inplace=True, errors="ignore")

# Step 7: Optimize memory usage
float_cols = mens_data.select_dtypes(include=['float64']).columns
mens_data[float_cols] = mens_data[float_cols].astype('float32')

# Step 8: Save the cleaned dataset
output_file = "cleaned_mens_data.csv"
mens_data.to_csv(output_file, index=False)
print(f"Cleaned men's data saved to {output_file}!")

# Step 9: Validate the cleaned dataset
print("Dataset preview:")
print(mens_data.head())

print("Dataset summary:")
print(mens_data.info())


Cleaned men's data saved to cleaned_mens_data.csv!
Dataset preview:
   Season  TeamID ConfAbbrev_x    TeamName  FirstD1Season  LastD1Season  \
0    1985    1102          wac   Air Force           1985          2025   
1    1985    1103          ovc       Akron           1985          2025   
2    1985    1104          sec     Alabama           1985          2025   
3    1985    1106         swac  Alabama St           1985          2025   
4    1985    1108         swac   Alcorn St           1985          2025   

   DayNum_x  WTeamID  WScore  LTeamID_x  ...  LTO LStl  LBlk  LPF  WTeamName  \
0       NaN      NaN     0.0        NaN  ...  NaN  NaN   NaN  NaN    Unknown   
1       NaN      NaN     0.0        NaN  ...  NaN  NaN   NaN  NaN    Unknown   
2       NaN      NaN     0.0        NaN  ...  NaN  NaN   NaN  NaN    Unknown   
3       NaN      NaN     0.0        NaN  ...  NaN  NaN   NaN  NaN    Unknown   
4       NaN      NaN     0.0        NaN  ...  NaN  NaN   NaN  NaN    Unknown   



In [33]:
print(regular_season_detailed_results)


        Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT  WFGM  \
0         2003      10     1104      68     1328      62    N      0    27   
1         2003      10     1272      70     1393      63    N      0    26   
2         2003      11     1266      73     1437      61    N      0    24   
3         2003      11     1296      56     1457      50    N      0    18   
4         2003      11     1400      77     1208      71    N      0    30   
...        ...     ...      ...     ...      ...     ...  ...    ...   ...   
117743    2025     106     1461      69     1102      62    H      0    25   
117744    2025     106     1462      76     1139      63    H      0    29   
117745    2025     106     1466      80     1480      62    H      0    28   
117746    2025     106     1468      94     1122      68    H      0    36   
117747    2025     106     1474      89     1146      72    H      0    28   

        WFGA  ...  LFGA3  LFTM  LFTA  LOR  LDR  LAst  LTO  LStl

In [ ]:
# Section 1.2: Load Womens the Data



KeyError: 'WTeamID'

In [ ]:








# Section 1.5: Load the Data
# Replace 'file_path' with the actual file paths for the datasets
mens_data = 
womens_data = 

# Combine the data if necessary
data = pd.concat([mens_data, womens_data], axis=0)

# Section 2: Exploratory Data Analysis (EDA)
# Summary statistics
print(data.describe())

# Visualizations for insights
plt.figure(figsize=(10, 6))
sns.countplot(x='team_won', data=data)
plt.title('Team Win Distribution')
plt.show()

# Section 3: Data Preprocessing
# Handle missing values
data.fillna(data.mean(), inplace=True)

# Feature Engineering
# Example: Calculate team performance over the last 5 games
data['last5_win_rate'] = data.groupby('team_id')['team_won'].rolling(window=5).mean().reset_index(0, drop=True)

# Encode categorical variables (if any)
data = pd.get_dummies(data, columns=['team_location'])

# Standardize features
scaler = StandardScaler()
numeric_columns = ['feature1', 'feature2']  # Replace with actual numeric columns
data[numeric_columns] = scaler.fit_transform(data[numeric_columns])

# Section 4: Model Training
# Split the dataset
X = data.drop(['game_id', 'team_won'], axis=1)  # Features
y = data['team_won']  # Target variable

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest Classifier (or any other model of your choice)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Section 5: Model Evaluation
# Predict on the test set
y_pred = model.predict_proba(X_test)[:, 1]

# Evaluate using the Brier Score
score = brier_score_loss(y_test, y_pred)
print(f"Brier Score: {score}")

# Section 6: Prepare Submission File
# Generate predictions for the 2025 tournament
predictions = model.predict_proba(X)[:, 1]

# Format the submission file
submission = pd.DataFrame({
    'ID': data['game_id'],  # Assuming game_id follows the required format
    'Pred': predictions
})
submission.to_csv('submission.csv', index=False)
print("Submission file created!")

# Section 7: Iterate and Improve
# Add notes or comments here for future iterations